# 📘 Async & Concurrent Python for Data Engineering
## Databricks Notebook — Parallel Processing Mastery

**What you'll learn:**
- Threading vs multiprocessing vs async (decision framework)
- concurrent.futures: ThreadPoolExecutor, ProcessPoolExecutor
- asyncio fundamentals: async/await, gather, semaphores
- Concurrent API ingestion (10x speedup)
- Producer-consumer pattern
- Bounded concurrency, circuit breakers, graceful shutdown

**Key rules:**
- Use `nest_asyncio` in Databricks (event loop already running)
- Always bound concurrency with Semaphore or max_workers
- Always set timeouts on concurrent operations
- ThreadPoolExecutor for I/O, ProcessPoolExecutor for CPU

**Prerequisites:** Notebooks 01-17

---

---
# 📗 Section 1: Why Concurrent Processing for DE

**The problem:** Sequential API calls are slow.
- 10 APIs × 2 seconds each = **20 seconds** sequential
- 10 APIs × 2 seconds each = **~2 seconds** concurrent (10x faster!)

**When concurrency helps (I/O-bound):**
- API calls (waiting for network)
- Database queries (waiting for DB)
- File reads/writes (waiting for disk)

**When it doesn't help (CPU-bound):**
- Data parsing, compression, hashing
- Use multiprocessing instead (bypasses GIL)

---
# 📗 Section 2: Threading vs Multiprocessing vs Async

| Feature | Threading | Multiprocessing | Async |
|---|---|---|---|
| Best for | I/O-bound | CPU-bound | I/O-bound (many) |
| Parallelism | Concurrent (GIL) | True parallel | Concurrent |
| Memory | Shared | Separate | Shared |
| Overhead | Low | High (process spawn) | Very low |
| Complexity | Medium | Medium | High |
| DE use case | API calls, DB queries | Data parsing, hashing | 100+ API calls |

**Decision framework:**
1. Few I/O tasks (< 20) → `ThreadPoolExecutor` (simplest)
2. Many I/O tasks (100+) → `asyncio` (most efficient)
3. CPU-heavy work → `ProcessPoolExecutor` (true parallelism)

---
# 📗 Section 3: concurrent.futures (Practical — Works in Databricks)

In [0]:
import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

def fetch_api(url):
    """Simulate an API call with network latency."""
    start = time.time()
    try:
        response = requests.get(url, timeout=10)
        return {"url": url, "status": response.status_code, "duration": time.time() - start}
    except Exception as e:
        return {"url": url, "status": "error", "duration": time.time() - start, "error": str(e)}

# URLs to fetch
urls = [f"https://jsonplaceholder.typicode.com/posts/{i}" for i in range(1, 11)]

# SEQUENTIAL (slow)
start = time.time()
sequential_results = [fetch_api(url) for url in urls]
sequential_time = time.time() - start

# CONCURRENT with ThreadPoolExecutor (fast)
start = time.time()
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(fetch_api, url): url for url in urls}
    concurrent_results = []
    for future in as_completed(futures):
        concurrent_results.append(future.result())
concurrent_time = time.time() - start

print(f"Sequential (10 calls): {sequential_time:.2f}s")
print(f"Concurrent (10 calls): {concurrent_time:.2f}s")
print(f"Speedup: {sequential_time/concurrent_time:.1f}x")

In [0]:
# ProcessPoolExecutor for CPU-bound work
from concurrent.futures import ProcessPoolExecutor
import hashlib
import time

def compute_hash(data):
    """CPU-intensive: compute hash of large string."""
    text = f"data_{data}" * 10000
    return hashlib.sha256(text.encode()).hexdigest()[:16]

items = list(range(100))

# Sequential
start = time.time()
seq_results = [compute_hash(i) for i in items]
seq_time = time.time() - start

# Parallel (ProcessPoolExecutor)
start = time.time()
with ProcessPoolExecutor(max_workers=4) as executor:
    par_results = list(executor.map(compute_hash, items))
par_time = time.time() - start

print(f"CPU-bound hashing (100 items):")
print(f"  Sequential: {seq_time:.2f}s")
print(f"  Parallel (4 workers): {par_time:.2f}s")
print(f"  Speedup: {seq_time/max(par_time, 0.001):.1f}x")

In [0]:
# ============================================
# 🎯 YOUR TURN: Parallel API Calls
# ============================================
# 1. Create a list of 5 URLs (jsonplaceholder posts 1-5)
# 2. Fetch them sequentially and time it
# 3. Fetch them with ThreadPoolExecutor(max_workers=5) and time it
# 4. Print the speedup factor
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
import time
import requests
from concurrent.futures import ThreadPoolExecutor

urls = [f"https://jsonplaceholder.typicode.com/posts/{i}" for i in range(1, 6)]

# Sequential
start = time.time()
for url in urls:
    requests.get(url, timeout=10)
seq_time = time.time() - start

# Concurrent
start = time.time()
with ThreadPoolExecutor(max_workers=5) as ex:
    list(ex.map(lambda u: requests.get(u, timeout=10), urls))
con_time = time.time() - start

print(f"Sequential: {seq_time:.2f}s")
print(f"Concurrent: {con_time:.2f}s")
print(f"Speedup: {seq_time/con_time:.1f}x")

---
# 📗 Section 4: asyncio Fundamentals

In [0]:
%pip install nest_asyncio --quiet

In [0]:
import asyncio
import nest_asyncio
import time

nest_asyncio.apply()  # Allow nested event loops in Databricks

# Basic async function
async def fetch_data(source_id, delay):
    """Simulate an async API call."""
    await asyncio.sleep(delay)  # Non-blocking wait
    return {"source": source_id, "records": source_id * 100, "delay": delay}

# Run multiple coroutines concurrently with gather
async def main():
    start = time.time()
    
    # These run CONCURRENTLY (not sequentially)
    results = await asyncio.gather(
        fetch_data(1, 1.0),
        fetch_data(2, 1.5),
        fetch_data(3, 0.8),
        fetch_data(4, 1.2),
        fetch_data(5, 0.5),
    )
    
    duration = time.time() - start
    print(f"All 5 sources fetched in {duration:.2f}s (max single delay was 1.5s)")
    for r in results:
        print(f"  Source {r['source']}: {r['records']} records ({r['delay']}s)")

asyncio.run(main())

---
# 📗 Section 5: Concurrent API Ingestion with Rate Limiting

In [0]:
import asyncio
import time
import random

# Semaphore limits concurrent operations (rate limiting)
async def rate_limited_fetch(semaphore, source_id):
    """Fetch with bounded concurrency."""
    async with semaphore:  # Only N can run at once
        delay = random.uniform(0.5, 2.0)
        await asyncio.sleep(delay)  # Simulate API call
        # Simulate occasional failures
        if random.random() < 0.1:
            raise Exception(f"Source {source_id} timeout")
        return {"source": source_id, "records": random.randint(100, 1000), "duration": delay}

async def ingest_all(num_sources=10, max_concurrent=3):
    """Ingest from multiple sources with rate limiting."""
    semaphore = asyncio.Semaphore(max_concurrent)
    
    tasks = [rate_limited_fetch(semaphore, i) for i in range(1, num_sources + 1)]
    
    # gather with return_exceptions=True (don't fail all if one fails)
    results = await asyncio.gather(*tasks, return_exceptions=True)
    
    successes = [r for r in results if isinstance(r, dict)]
    failures = [r for r in results if isinstance(r, Exception)]
    
    return successes, failures

start = time.time()
successes, failures = asyncio.run(ingest_all(10, max_concurrent=3))
duration = time.time() - start

print(f"Concurrent ingestion (10 sources, max 3 concurrent):")
print(f"  Duration: {duration:.2f}s")
print(f"  Successes: {len(successes)}")
print(f"  Failures: {len(failures)}")
if failures:
    for f in failures:
        print(f"    ❌ {f}")

---
# 📗 Section 6: Producer-Consumer Pattern

In [0]:
import asyncio
import random

async def producer(queue, num_items):
    """Produce items (simulate fetching API pages)."""
    for i in range(num_items):
        await asyncio.sleep(random.uniform(0.1, 0.3))  # Simulate fetch
        item = {"page": i + 1, "records": random.randint(50, 200)}
        await queue.put(item)
        print(f"  [Producer] Page {i+1} fetched ({item['records']} records)")
    
    # Signal done (poison pill)
    await queue.put(None)

async def consumer(queue, consumer_id):
    """Consume items (simulate processing and writing)."""
    processed = 0
    while True:
        item = await queue.get()
        if item is None:
            await queue.put(None)  # Pass poison pill to next consumer
            break
        await asyncio.sleep(0.1)  # Simulate processing
        processed += item["records"]
        print(f"  [Consumer {consumer_id}] Processed page {item['page']}")
    return processed

async def pipeline():
    """Producer-consumer pipeline."""
    queue = asyncio.Queue(maxsize=5)  # Bounded queue (backpressure)
    
    # Start producer and consumers
    producer_task = asyncio.create_task(producer(queue, 8))
    consumer_tasks = [
        asyncio.create_task(consumer(queue, i)) for i in range(1, 3)
    ]
    
    await producer_task
    results = await asyncio.gather(*consumer_tasks)
    print(f"\n  Total processed: {sum(results)} records")

print("Producer-Consumer Pipeline:")
asyncio.run(pipeline())

---
# 📗 Section 7: Production Patterns

In [0]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import random

class CircuitBreaker:
    """Stop calling a failing service after N consecutive failures."""
    
    def __init__(self, failure_threshold=3, cooldown_seconds=10):
        self.failure_threshold = failure_threshold
        self.cooldown_seconds = cooldown_seconds
        self.consecutive_failures = 0
        self.last_failure_time = 0
        self.state = "closed"  # closed=normal, open=blocking, half_open=testing
    
    def can_proceed(self):
        if self.state == "closed":
            return True
        if self.state == "open":
            if time.time() - self.last_failure_time > self.cooldown_seconds:
                self.state = "half_open"
                return True
            return False
        return True  # half_open: allow one attempt
    
    def record_success(self):
        self.consecutive_failures = 0
        self.state = "closed"
    
    def record_failure(self):
        self.consecutive_failures += 1
        self.last_failure_time = time.time()
        if self.consecutive_failures >= self.failure_threshold:
            self.state = "open"

# Demo: concurrent processing with circuit breaker
breaker = CircuitBreaker(failure_threshold=3, cooldown_seconds=5)

def process_with_breaker(item_id):
    if not breaker.can_proceed():
        return {"id": item_id, "status": "skipped (circuit open)"}
    
    try:
        # Simulate: items 5-7 always fail
        if 5 <= item_id <= 7:
            raise ConnectionError(f"Service unavailable for item {item_id}")
        time.sleep(0.1)
        breaker.record_success()
        return {"id": item_id, "status": "success"}
    except Exception as e:
        breaker.record_failure()
        return {"id": item_id, "status": f"failed: {e}"}

print("Circuit Breaker Demo:")
with ThreadPoolExecutor(max_workers=3) as executor:
    futures = [executor.submit(process_with_breaker, i) for i in range(1, 12)]
    for f in as_completed(futures):
        r = f.result()
        icon = "✅" if r["status"] == "success" else "⚠️" if "skipped" in r["status"] else "❌"
        print(f"  {icon} Item {r['id']}: {r['status']}")

print(f"\nCircuit state: {breaker.state} (failures: {breaker.consecutive_failures})")

---
# 🚀 Section 8: Mini Projects

## Project 1: Concurrent Data Ingestion System

In [0]:
import time
import random
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd

def simulate_api_call(endpoint_id):
    """Simulate API call with variable latency."""
    delay = random.uniform(0.5, 2.0)
    time.sleep(delay)
    # Simulate occasional failure
    if random.random() < 0.15:
        raise Exception(f"Endpoint {endpoint_id} timeout")
    return {
        "endpoint_id": endpoint_id,
        "records": random.randint(50, 500),
        "duration_s": round(delay, 2)
    }

def ingest_concurrent(num_endpoints=10, max_workers=5, max_retries=2):
    """Concurrent ingestion with retries."""
    results = []
    failed = []
    
    # First attempt
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(simulate_api_call, i): i for i in range(1, num_endpoints + 1)}
        for future in as_completed(futures):
            endpoint_id = futures[future]
            try:
                results.append(future.result())
            except Exception as e:
                failed.append(endpoint_id)
    
    # Retry failed endpoints
    for retry in range(max_retries):
        if not failed:
            break
        retry_failed = []
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(simulate_api_call, eid): eid for eid in failed}
            for future in as_completed(futures):
                eid = futures[future]
                try:
                    results.append(future.result())
                except Exception:
                    retry_failed.append(eid)
        failed = retry_failed
    
    return results, failed

# Run and compare
print("=== SEQUENTIAL ===")
start = time.time()
seq_results = []
for i in range(1, 11):
    try:
        seq_results.append(simulate_api_call(i))
    except:
        pass
seq_time = time.time() - start
print(f"  Duration: {seq_time:.2f}s, Success: {len(seq_results)}")

print("\n=== CONCURRENT ===")
start = time.time()
con_results, con_failed = ingest_concurrent(10, max_workers=5)
con_time = time.time() - start
print(f"  Duration: {con_time:.2f}s, Success: {len(con_results)}, Failed: {len(con_failed)}")
print(f"  Speedup: {seq_time/con_time:.1f}x")

# Write to Delta
if con_results:
    pdf = pd.DataFrame(con_results)
    spark.createDataFrame(pdf).write.format("delta").mode("overwrite").saveAsTable("techmart_dw.concurrent_ingestion_results")
    print(f"\n✅ Written {len(con_results)} results to Delta")

---
# 🏆 Section 9: Interview Questions

## Q1: Async vs threading vs multiprocessing?

- **Threading:** I/O-bound, simple, GIL limits CPU parallelism. Use for < 20 concurrent tasks.
- **Async:** I/O-bound, single thread, very efficient for 100+ concurrent tasks. Higher complexity.
- **Multiprocessing:** CPU-bound, true parallelism, separate memory. Use for data parsing, hashing.

## Q2: Rate limiting with concurrent API calls?

Use `asyncio.Semaphore(N)` or `ThreadPoolExecutor(max_workers=N)` to limit concurrent requests.
Also respect `Retry-After` headers and implement exponential backoff.

## Q3: Prevent overwhelming a source system?

1. **Bounded concurrency:** max_workers or Semaphore
2. **Rate limiting:** sleep between batches
3. **Circuit breaker:** stop after N consecutive failures
4. **Backpressure:** bounded queues prevent runaway producers

## Q4: GIL and its impact?

The Global Interpreter Lock prevents multiple threads from executing Python bytecode simultaneously.
- **I/O-bound:** GIL is released during I/O waits → threading works fine
- **CPU-bound:** GIL blocks parallelism → use multiprocessing instead

## Q5: Producer-consumer pattern?

Producer fetches data → puts in queue → Consumer processes and writes.
Use when: production rate != consumption rate, need buffering, want to decouple stages.
Bounded queue provides backpressure (producer waits if queue is full).

## Q6: Partial failures (3 of 10 fail)?

Use `return_exceptions=True` in `asyncio.gather()` or catch exceptions per-future in `as_completed()`.
Collect successes and failures separately. Retry failures. Report both.

## Q7: gather() vs as_completed()?

- `gather()`: Waits for ALL to finish, returns results in order.
- `as_completed()`: Yields results as each finishes (out of order). Better for progress reporting.

## Q8: Backpressure to prevent memory overflow?

Use bounded queues (`asyncio.Queue(maxsize=N)` or `queue.Queue(maxsize=N)`).
When queue is full, producer blocks until consumer catches up. Prevents unbounded memory growth.

---
# 📗 Section 4: Async Patterns for DE Pipelines

Async is most valuable in DE when you have I/O-bound work: API calls,
database queries, file reads. Here are the production patterns.

In [0]:
import asyncio
import time
import random
from typing import List, Dict, Any, Callable, Awaitable
from dataclasses import dataclass, field
from datetime import datetime

# ============================================================
# PATTERN 1: Async API Ingestion with Rate Limiting
# ============================================================

class AsyncRateLimiter:
    """
    Token bucket rate limiter for async API calls.
    Prevents hitting API rate limits.
    """
    
    def __init__(self, calls_per_second: float):
        self.calls_per_second = calls_per_second
        self.min_interval = 1.0 / calls_per_second
        self._last_call = 0
        self._lock = asyncio.Lock()
    
    async def acquire(self):
        async with self._lock:
            now = time.monotonic()
            elapsed = now - self._last_call
            if elapsed < self.min_interval:
                await asyncio.sleep(self.min_interval - elapsed)
            self._last_call = time.monotonic()

async def fetch_with_rate_limit(
    endpoint_id: int,
    rate_limiter: AsyncRateLimiter,
    simulate_latency: float = 0.1,
) -> dict:
    """Simulate an API call with rate limiting."""
    await rate_limiter.acquire()
    
    # Simulate network latency
    await asyncio.sleep(simulate_latency + random.uniform(0, 0.05))
    
    # Simulate occasional failures
    if random.random() < 0.1:
        raise ConnectionError(f"API timeout for endpoint {endpoint_id}")
    
    return {
        "id": endpoint_id,
        "data": f"result_{endpoint_id}",
        "fetched_at": datetime.utcnow().isoformat(),
    }

async def ingest_with_rate_limiting(endpoint_ids: List[int],
                                     calls_per_second: float = 5.0) -> dict:
    """Ingest from multiple endpoints with rate limiting."""
    limiter = AsyncRateLimiter(calls_per_second)
    results = {"success": [], "errors": []}
    
    tasks = [fetch_with_rate_limit(eid, limiter) for eid in endpoint_ids]
    
    # gather with return_exceptions=True to handle partial failures
    responses = await asyncio.gather(*tasks, return_exceptions=True)
    
    for eid, response in zip(endpoint_ids, responses):
        if isinstance(response, Exception):
            results["errors"].append({"id": eid, "error": str(response)})
        else:
            results["success"].append(response)
    
    return results

# Run it
async def demo_rate_limiting():
    start = time.time()
    result = await ingest_with_rate_limiting(list(range(1, 11)), calls_per_second=10.0)
    elapsed = time.time() - start
    print(f"Ingested {len(result['success'])} endpoints in {elapsed:.2f}s")
    print(f"Errors: {len(result['errors'])}")
    if result["errors"]:
        print(f"Error details: {result['errors']}")

asyncio.run(demo_rate_limiting())

In [0]:
# ============================================================
# PATTERN 2: Async Pipeline with Semaphore (concurrency control)
# ============================================================

async def process_file(file_path: str, semaphore: asyncio.Semaphore,
                        simulate_work: float = 0.05) -> dict:
    """Process a single file with concurrency control."""
    async with semaphore:  # Limit concurrent file operations
        await asyncio.sleep(simulate_work)  # Simulate I/O
        return {
            "file": file_path,
            "size_bytes": random.randint(1000, 1_000_000),
            "rows": random.randint(100, 100_000),
            "processed_at": datetime.utcnow().isoformat(),
        }

async def process_file_batch(
    file_paths: List[str],
    max_concurrent: int = 5,
) -> List[dict]:
    """
    Process files concurrently with a semaphore to limit parallelism.
    
    max_concurrent=5 means at most 5 files processed simultaneously.
    This prevents overwhelming the filesystem or downstream systems.
    """
    semaphore = asyncio.Semaphore(max_concurrent)
    
    print(f"Processing {len(file_paths)} files (max {max_concurrent} concurrent)...")
    start = time.time()
    
    tasks = [process_file(fp, semaphore) for fp in file_paths]
    results = await asyncio.gather(*tasks, return_exceptions=True)
    
    elapsed = time.time() - start
    successful = [r for r in results if not isinstance(r, Exception)]
    failed = [r for r in results if isinstance(r, Exception)]
    
    print(f"  ✅ Processed {len(successful)} files in {elapsed:.2f}s")
    print(f"  ❌ Failed: {len(failed)}")
    
    total_rows = sum(r["rows"] for r in successful)
    total_bytes = sum(r["size_bytes"] for r in successful)
    print(f"  Total rows: {total_rows:,}, Total bytes: {total_bytes:,}")
    
    return successful

# Compare sequential vs concurrent
async def benchmark_concurrency():
    files = [f"s3://bucket/data/file_{i:04d}.parquet" for i in range(20)]
    
    # Sequential (max_concurrent=1)
    start = time.time()
    await process_file_batch(files, max_concurrent=1)
    seq_time = time.time() - start
    
    # Concurrent (max_concurrent=10)
    start = time.time()
    await process_file_batch(files, max_concurrent=10)
    conc_time = time.time() - start
    
    print(f"\nSequential: {seq_time:.2f}s")
    print(f"Concurrent: {conc_time:.2f}s")
    print(f"Speedup: {seq_time/conc_time:.1f}x")

asyncio.run(benchmark_concurrency())

In [0]:
# ============================================================
# PATTERN 3: Async Producer-Consumer Pipeline
# ============================================================

async def producer(queue: asyncio.Queue, items: List[Any],
                   name: str = "producer"):
    """Produces items and puts them in the queue."""
    for item in items:
        await asyncio.sleep(0.01)  # Simulate data arrival
        await queue.put(item)
    
    # Signal completion with sentinel values
    await queue.put(None)
    print(f"  {name}: produced {len(items)} items")

async def consumer(queue: asyncio.Queue, results: list,
                   transform_fn: Callable, name: str = "consumer"):
    """Consumes items from queue and applies transformation."""
    processed = 0
    while True:
        item = await queue.get()
        
        if item is None:  # Sentinel — done
            queue.task_done()
            break
        
        try:
            result = await transform_fn(item)
            results.append(result)
            processed += 1
        except Exception as e:
            print(f"  {name} error: {e}")
        finally:
            queue.task_done()
    
    print(f"  {name}: processed {processed} items")

async def transform_order(order: dict) -> dict:
    """Async transformation: enrich order with computed fields."""
    await asyncio.sleep(0.005)  # Simulate async DB lookup
    return {
        **order,
        "tax": round(order["amount"] * 0.08, 2),
        "total": round(order["amount"] * 1.08, 2),
        "tier": "high" if order["amount"] > 200 else "standard",
    }

async def run_pipeline():
    """Run a producer-consumer pipeline."""
    orders = [
        {"order_id": i, "amount": random.uniform(50, 500)}
        for i in range(1, 31)
    ]
    
    queue = asyncio.Queue(maxsize=10)  # Backpressure: max 10 items buffered
    results = []
    
    # Run producer and 3 consumers concurrently
    await asyncio.gather(
        producer(queue, orders),
        consumer(queue, results, transform_order, "consumer-1"),
        consumer(queue, results, transform_order, "consumer-2"),
        consumer(queue, results, transform_order, "consumer-3"),
    )
    
    print(f"\nPipeline complete: {len(results)} orders processed")
    print(f"Total revenue: ${sum(r['total'] for r in results):,.2f}")
    print(f"High-tier orders: {sum(1 for r in results if r['tier'] == 'high')}")

asyncio.run(run_pipeline())

---
# 📗 Section 5: Practice Exercises

In [0]:
# ============================================================
# EXERCISE 1: Async Multi-Source Data Aggregator
# ============================================================
# Fetch data from multiple "sources" concurrently and merge

async def fetch_source(source_name: str, delay: float) -> dict:
    """Simulate fetching from a data source."""
    await asyncio.sleep(delay)
    return {
        "source": source_name,
        "records": random.randint(100, 10000),
        "schema_version": random.choice(["v1", "v2"]),
        "fetched_at": datetime.utcnow().isoformat(),
    }

async def aggregate_sources(sources: Dict[str, float]) -> dict:
    """
    Fetch from all sources concurrently.
    sources: {source_name: simulated_delay}
    """
    start = time.time()
    
    tasks = {
        name: asyncio.create_task(fetch_source(name, delay))
        for name, delay in sources.items()
    }
    
    results = {}
    errors = {}
    
    for name, task in tasks.items():
        try:
            results[name] = await task
        except Exception as e:
            errors[name] = str(e)
    
    elapsed = time.time() - start
    total_records = sum(r["records"] for r in results.values())
    
    print(f"Aggregated {len(results)} sources in {elapsed:.2f}s")
    print(f"Total records: {total_records:,}")
    for name, result in results.items():
        print(f"  {name}: {result['records']:,} records ({result['schema_version']})")
    
    return {"results": results, "errors": errors, "total_records": total_records}

# Sources with different latencies
data_sources = {
    "orders_api": 0.3,
    "customers_db": 0.1,
    "products_catalog": 0.2,
    "inventory_service": 0.4,
    "analytics_warehouse": 0.5,
}

result = asyncio.run(aggregate_sources(data_sources))
print(f"\nMax sequential time would be: {sum(data_sources.values()):.1f}s")

---
# 📗 Section 5: Async Patterns for Data Engineering

## When to Use Async in DE

Async is useful when you have I/O-bound operations that can run concurrently:

| Use Case | Sync | Async | Speedup |
|----------|------|-------|---------|
| Fetch 100 API endpoints | 100 * 1s = 100s | ~1-2s | 50-100x |
| Write to 10 databases | 10 * 2s = 20s | ~2-3s | 7-10x |
| Process 1000 files | 1000 * 0.1s = 100s | ~5-10s | 10-20x |
| CPU-heavy transforms | 100s | 100s | 1x (no benefit) |

## Async API Calls

```python
import asyncio
import aiohttp

async def fetch_one(session, url, params=None):
    async with session.get(url, params=params) as response:
        return await response.json()

async def fetch_all(urls):
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_one(session, url) for url in urls]
        return await asyncio.gather(*tasks)

# Fetch 100 endpoints concurrently
urls = [f"https://api.example.com/orders/{i}" for i in range(100)]
results = asyncio.run(fetch_all(urls))
```

## Async with Rate Limiting

```python
import asyncio
import aiohttp
from asyncio import Semaphore

async def fetch_with_limit(session, url, semaphore):
    async with semaphore:  # Limit concurrent requests
        async with session.get(url) as response:
            return await response.json()

async def fetch_all_limited(urls, max_concurrent=10):
    semaphore = Semaphore(max_concurrent)
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_with_limit(session, url, semaphore) for url in urls]
        return await asyncio.gather(*tasks)
```

In [0]:
# Async patterns for data engineering
import asyncio
import time
from concurrent.futures import ThreadPoolExecutor

print("Async Patterns for Data Engineering")
print("=" * 60)

# Simulate I/O-bound operations
async def fetch_data(source_id, delay=0.1):
    """Simulate fetching data from an API."""
    await asyncio.sleep(delay)  # Simulate network I/O
    return {"source": source_id, "records": source_id * 100}

async def process_sources_async(source_ids):
    """Fetch all sources concurrently."""
    tasks = [fetch_data(sid) for sid in source_ids]
    results = await asyncio.gather(*tasks)
    return results

# Compare sync vs async
source_ids = list(range(1, 11))  # 10 sources

print("\n1. Sequential (sync) approach:")
start = time.time()
sync_results = []
for sid in source_ids:
    # Simulate blocking I/O
    time.sleep(0.01)  # 10ms per source
    sync_results.append({"source": sid, "records": sid * 100})
sync_time = time.time() - start
print(f"   Time: {sync_time:.3f}s for {len(source_ids)} sources")

print("\n2. Concurrent (async) approach:")
start = time.time()
async_results = asyncio.run(process_sources_async(source_ids))
async_time = time.time() - start
print(f"   Time: {async_time:.3f}s for {len(source_ids)} sources")
print(f"   Speedup: {sync_time/async_time:.1f}x")

print("\n3. Results (first 3):")
for r in async_results[:3]:
    print(f"   {r}")

print("\n4. When to use async in DE:")
use_cases = [
    "Fetching data from multiple REST APIs simultaneously",
    "Writing to multiple databases/tables concurrently",
    "Processing multiple files in parallel",
    "Sending notifications to multiple channels",
    "Health checks across multiple services",
]
for uc in use_cases:
    print(f"   * {uc}")

print("\n5. When NOT to use async:")
not_use = [
    "CPU-heavy computations (use multiprocessing instead)",
    "Spark transformations (Spark handles parallelism internally)",
    "Simple sequential pipelines (adds complexity for no benefit)",
]
for nu in not_use:
    print(f"   * {nu}")


---
# ✅ Notebook Complete!

**What was covered:**
- Threading vs multiprocessing vs async decision framework
- concurrent.futures: ThreadPoolExecutor, ProcessPoolExecutor, as_completed
- asyncio: async/await, gather, Semaphore for rate limiting
- Concurrent API ingestion with timing comparison (10x speedup)
- Producer-consumer pattern with bounded queues
- Production patterns: circuit breaker, retries, bounded concurrency
- Mini project: concurrent ingestion system with retries and Delta output
- 8 interview questions

**Next:** Notebook 19 — Pytest for Data Engineering

---